In [1]:
# CELL 1 — Imports & config

import requests
import os
import json
import time
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

NVD_API_KEY = os.getenv("NVD_API_KEY")
BASE_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
HEADERS = {"apiKey": NVD_API_KEY}
RESULTS_PER_PAGE = 2000  # NVD max allowed per request
DATA_DIR = "../data"

os.makedirs(DATA_DIR, exist_ok=True)
print(f"API Key loaded: {'✅' if NVD_API_KEY else '❌ NOT FOUND'}")


API Key loaded: ✅


In [2]:
#CELL 2 — Fetch one page, inspect what we get back
# Before pulling everything, let's understand the structure

params = {
    "resultsPerPage": 5,
    "startIndex": 0,
    "cvssV3Severity": "CRITICAL"  # test with CRITICAL first
}

response = requests.get(BASE_URL, headers=HEADERS, params=params)
data = response.json()

print(f"Status: {response.status_code}")
print(f"Total CRITICAL CVEs available: {data['totalResults']}")
print(f"\nSample CVE ID: {data['vulnerabilities'][0]['cve']['id']}")

Status: 200
Total CRITICAL CVEs available: 29188

Sample CVE ID: CVE-1999-0426


In [3]:
#CELL 3 — Full paginated fetcher function
# NVD only returns 2000 results per request, so we need to
# loop through pages using startIndex until we have everything

def fetch_cves(severity, max_cves=5000):
    """
    Fetch CVEs by severity from NVD API using pagination.
    severity: "HIGH" or "CRITICAL"
    max_cves: cap on how many to fetch (for testing, lower this)
    """
    all_cves = []
    start = 0

    # First request to get total count
    params = {
        "resultsPerPage": 1,
        "startIndex": 0,
        "cvssV3Severity": severity
    }
    r = requests.get(BASE_URL, headers=HEADERS, params=params)
    total = r.json()["totalResults"]
    total_to_fetch = min(total, max_cves)

    print(f"Fetching {total_to_fetch} {severity} CVEs (total available: {total})")

    with tqdm(total=total_to_fetch) as pbar:
        while start < total_to_fetch:
            params = {
                "resultsPerPage": RESULTS_PER_PAGE,
                "startIndex": start,
                "cvssV3Severity": severity
            }
            r = requests.get(BASE_URL, headers=HEADERS, params=params)

            if r.status_code != 200:
                print(f"❌ Error at startIndex {start}: {r.status_code}")
                break

            batch = r.json().get("vulnerabilities", [])
            all_cves.extend(batch)
            pbar.update(len(batch))
            start += len(batch)

            # NVD rate limit: 50 requests per 30 seconds with API key
            # Being conservative with 0.6s delay between requests
            time.sleep(0.6)

    print(f"✅ Fetched {len(all_cves)} {severity} CVEs")
    return all_cves


In [4]:
#CELL 4 — Test fetch with a small number first (50 CVEs)
# Always test small before running the full pull

test_critical = fetch_cves("CRITICAL", max_cves=50)
print(f"\nSample keys: {list(test_critical[0]['cve'].keys())}")

Fetching 50 CRITICAL CVEs (total available: 29188)


2000it [00:02, 837.32it/s]                           

✅ Fetched 2000 CRITICAL CVEs

Sample keys: ['id', 'sourceIdentifier', 'published', 'lastModified', 'vulnStatus', 'cveTags', 'descriptions', 'metrics', 'weaknesses', 'configurations', 'references']


In [5]:
#CELL 5 — Cleaning function
# Filter out CVEs that are incomplete or unusable for our purposes

def clean_cve(raw):
    """
    Extract and validate fields we need from a raw NVD CVE object.
    Returns a clean dict or None if the CVE fails quality checks.
    """
    cve = raw["cve"]
    metrics = cve.get("metrics", {})

    # Must have CVSS v3.1
    if "cvssMetricV31" not in metrics:
        return None

    cvss = metrics["cvssMetricV31"][0]["cvssData"]

    # Must have a description in English
    descriptions = [d for d in cve.get("descriptions", []) if d["lang"] == "en"]
    if not descriptions or len(descriptions[0]["value"]) < 100:
        return None

    # Must have CPE (affected product info)
    configurations = cve.get("configurations", [])
    if not configurations:
        return None

    return {
        "id": cve["id"],
        "published": cve["published"],
        "lastModified": cve["lastModified"],
        "description": descriptions[0]["value"],
        "cvss_score": cvss["baseScore"],
        "cvss_severity": cvss["baseSeverity"],
        "attack_vector": cvss["attackVector"],
        "attack_complexity": cvss["attackComplexity"],
        "privileges_required": cvss["privilegesRequired"],
        "user_interaction": cvss["userInteraction"],
        "confidentiality_impact": cvss["confidentialityImpact"],
        "integrity_impact": cvss["integrityImpact"],
        "availability_impact": cvss["availabilityImpact"],
        "configurations": configurations,
        "references": [r["url"] for r in cve.get("references", [])]
    }

In [6]:
# CELL 6 — Test the cleaning function on our 50 sample CVEs
cleaned_sample = [clean_cve(c) for c in test_critical]
cleaned_sample = [c for c in cleaned_sample if c is not None]

total = len(test_critical)
passed = len(cleaned_sample)
print(f"Passed quality filter: {passed}/{total} ({round(passed/total*100)}%)")
print(f"\nSample cleaned CVE:")
print(json.dumps(cleaned_sample[0], indent=2))

Passed quality filter: 403/2000 (20%)

Sample cleaned CVE:
{
  "id": "CVE-1999-1324",
  "published": "1999-12-31T05:00:00.000",
  "lastModified": "2025-04-03T01:03:51.193",
  "description": "VAXstations running Open VMS 5.3 through 5.5-2 with VMS DECwindows or MOTIF do not properly disable access to user accounts that exceed the break-in limit threshold for failed login attempts, which makes it easier for attackers to conduct brute force password guessing.",
  "cvss_score": 9.8,
  "cvss_severity": "CRITICAL",
  "attack_vector": "NETWORK",
  "attack_complexity": "LOW",
  "privileges_required": "NONE",
  "user_interaction": "NONE",
  "confidentiality_impact": "HIGH",
  "integrity_impact": "HIGH",
  "availability_impact": "HIGH",
  "configurations": [
    {
      "nodes": [
        {
          "operator": "OR",
          "negate": false,
          "cpeMatch": [
            {
              "vulnerable": true,
              "criteria": "cpe:2.3:o:hp:openvms_vax:*:*:*:*:*:*:*:*",
           

In [9]:
# CELL 7 — Load into pandas for easier inspection

import pandas as pd

df = pd.DataFrame(cleaned_sample)

# Drop nested columns for display purposes (we keep them in the JSON)
df_display = df.drop(columns=["configurations", "references"])

print(f"Shape: {df_display.shape}")
print(f"\nColumn types:\n{df_display.dtypes}")
print(f"\nSeverity distribution:\n{df_display['cvss_severity'].value_counts()}")
print(f"\nAttack vector distribution:\n{df_display['attack_vector'].value_counts()}")

Shape: (403, 13)

Column types:
id                            str
published                     str
lastModified                  str
description                   str
cvss_score                float64
cvss_severity                 str
attack_vector                 str
attack_complexity             str
privileges_required           str
user_interaction              str
confidentiality_impact        str
integrity_impact              str
availability_impact           str
dtype: object

Severity distribution:
cvss_severity
CRITICAL    401
MEDIUM        2
Name: count, dtype: int64

Attack vector distribution:
attack_vector
NETWORK             400
LOCAL                 2
ADJACENT_NETWORK      1
Name: count, dtype: int64


In [10]:
# Display first 5 rows nicely
df_display.head()
cleaned_sample = [clean_cve(c) for c in test_critical]
cleaned_sample = [c for c in cleaned_sample if c is not None]

total = len(test_critical)
passed = len(cleaned_sample)
print(f"Passed quality filter: {passed}/{total} ({round(passed/total*100)}%)")
print(f"\nSample cleaned CVE:")
print(json.dumps(cleaned_sample[0], indent=2))

Passed quality filter: 403/2000 (20%)

Sample cleaned CVE:
{
  "id": "CVE-1999-1324",
  "published": "1999-12-31T05:00:00.000",
  "lastModified": "2025-04-03T01:03:51.193",
  "description": "VAXstations running Open VMS 5.3 through 5.5-2 with VMS DECwindows or MOTIF do not properly disable access to user accounts that exceed the break-in limit threshold for failed login attempts, which makes it easier for attackers to conduct brute force password guessing.",
  "cvss_score": 9.8,
  "cvss_severity": "CRITICAL",
  "attack_vector": "NETWORK",
  "attack_complexity": "LOW",
  "privileges_required": "NONE",
  "user_interaction": "NONE",
  "confidentiality_impact": "HIGH",
  "integrity_impact": "HIGH",
  "availability_impact": "HIGH",
  "configurations": [
    {
      "nodes": [
        {
          "operator": "OR",
          "negate": false,
          "cpeMatch": [
            {
              "vulnerable": true,
              "criteria": "cpe:2.3:o:hp:openvms_vax:*:*:*:*:*:*:*:*",
           

In [11]:
df_display.head()

,id,published,lastModified,description,cvss_score,cvss_severity,attack_vector,attack_complexity,privileges_required,user_interaction,confidentiality_impact,integrity_impact,availability_impact
0,CVE-1999-1324,1999-12-31T05:00:00.000,2025-04-03T01:03:51.193,VAXstations running Open VMS 5.3 through 5.5-2...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
1,CVE-2000-1218,2000-04-14T04:00:00.000,2025-04-03T01:03:51.193,The default configuration for the domain name ...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
2,CVE-2000-0944,2000-12-19T05:00:00.000,2025-04-03T01:03:51.193,CGI Script Center News Update 1.1 does not pro...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
3,CVE-2001-1339,2001-05-24T04:00:00.000,2025-04-03T01:03:51.193,Beck IPC GmbH IPC@CHIP telnet service does not...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
4,CVE-2001-0248,2001-06-18T04:00:00.000,2025-04-03T01:03:51.193,Buffer overflow in FTP server in HPUX 11 allow...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH
